# Stroke Prediction

## Importing necessary libraries

In [1]:
import numpy as np
import pandas as pd 

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import MinMaxScaler
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, fbeta_score, make_scorer

## Import dataset

In [2]:
df = pd.read_csv('../dataset/healthcare-dataset-stroke-data.csv')

In [3]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [5]:
df.describe()

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


In [6]:
df.describe(include=[object])

,gender,ever_married,work_type,Residence_type,smoking_status
count,5110,5110,5110,5110,5110
unique,3,2,5,2,4
top,Female,Yes,Private,Urban,never smoked
freq,2994,3353,2925,2596,1892


## Rename column for consistent formatting of name

In [7]:
df.rename(columns={'Residence_type': 'residence_type'}, inplace=True)

In [8]:
pd.set_option('display.max_columns', None)

In [9]:
pd.set_option('display.max_rows', None)

### Impute missing values in BMI column
For 201 entries missing BMI values, these entries will be imputed with the median BMI value

In [10]:
median_bmi = df['bmi'].median()
df['bmi'].fillna(median_bmi, inplace=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_644\2337012406.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['bmi'].fillna(median_bmi, inplace=True)


In [11]:
print(df['bmi'].isna().sum())

0


## Feature Engineering

### Converting categorical columns to numerical columns

In [12]:
df['male'] = (df['gender'] == 'Male').astype(int)

In [13]:
df['ever_married'] = (df['ever_married'] == 'Yes').astype(int)

In [14]:
df['urban'] = (df['residence_type'] == 'Urban').astype(int)

In [15]:
df['smoke'] = ((df['smoking_status'] == 'formerly smoked') | (df['smoking_status'] == 'smokes')).astype(int)

In [16]:
df['have_work'] = ((df['work_type'] == 'Govt_job') | (df['work_type'] == 'Private') | (df['work_type'] == 'Self-employed')).astype(int)

In [17]:
df['hypertension_or_heart_disease'] = df['hypertension'] | df['heart_disease']

In [18]:
df = pd.get_dummies(df, columns=['work_type', 'smoking_status'], drop_first = False)

### Extra columns for interaction features

In [19]:
df['age_ever_married'] = df['age'] * df['ever_married']

In [20]:
df['age_hypertension'] = df['age'] * df['hypertension']

In [21]:
df['age_heart_disease'] = df['age'] * df['heart_disease']

In [22]:
df['age_bmi'] = df['age'] * df['bmi']

In [23]:
df['age_avg_glucose_level'] = df['age'] * df['avg_glucose_level']

In [24]:
df['bmi_avg_glucose_level'] = df['bmi'] * df['avg_glucose_level']

In [25]:
df['avg_glucose_level_hypertension'] = df['avg_glucose_level'] * df['hypertension']

### Creating columns representing outstanding values for BMI and Average Glucose Level

In [26]:
df['obese'] = df['bmi'] >= 30

In [27]:
df['high_avg_glucose_level'] = df['avg_glucose_level'] >= 150

In [28]:
df['obese_high_avg_glucose_level'] = df['obese'] | df['high_avg_glucose_level']

### Creating column representing high age

In [29]:
df['old'] = df['age'] >= 65

### Converting boolean columns to numerical

In [30]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

### Accounting for multiple risks

In [31]:
df['risk_count'] = df['obese'] + df['high_avg_glucose_level'] + df['old'] + df['hypertension'] + df['heart_disease']

### Reformatting column names

In [32]:
df.rename(columns={'work_type_Govt_job': 'work_type_govt_job', 'work_type_Never_worked': 'work_type_never_worked', 'work_type_Private': 'work_type_private', 'work_type_Self-employed': 'work_type_self_employed'}, inplace=True)

In [33]:
df.rename(columns={'smoking_status_Unknown': 'smoking_status_unknown', 'smoking_status_formerly smoked': 'smoking_status_formerly_smoked', 'smoking_status_never smoked': 'smoking_status_never_smoked'}, inplace=True)

In [34]:
df.drop(columns=['gender', 'residence_type'], inplace=True)

### Log Transforming BMI and Average Glucose Level columns

In [35]:
df['log_bmi'] = np.log(df['bmi'])

In [36]:
df['log_avg_glucose_level'] = np.log(df['avg_glucose_level'])

In [37]:
df.head()

,id,age,hypertension,heart_disease,ever_married,avg_glucose_level,bmi,stroke,male,urban,smoke,have_work,hypertension_or_heart_disease,work_type_govt_job,work_type_never_worked,work_type_private,work_type_self_employed,work_type_children,smoking_status_unknown,smoking_status_formerly_smoked,smoking_status_never_smoked,smoking_status_smokes,age_ever_married,age_hypertension,age_heart_disease,age_bmi,age_avg_glucose_level,bmi_avg_glucose_level,avg_glucose_level_hypertension,obese,high_avg_glucose_level,obese_high_avg_glucose_level,old,risk_count,log_bmi,log_avg_glucose_level
0,9046,67.0,0,1,1,228.69,36.6,1,1,1,1,1,1,0,0,1,0,0,0,1,0,0,67.0,0.0,67.0,2452.2,15322.23,8370.054,0.00,1,1,1,1,4,3.600048,5.432367
1,51676,61.0,0,0,1,202.21,28.1,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,61.0,0.0,0.0,1714.1,12334.81,5682.101,0.00,0,1,1,0,1,3.335770,5.309307
2,31112,80.0,0,1,1,105.92,32.5,1,1,0,0,1,1,0,0,1,0,0,0,0,1,0,80.0,0.0,80.0,2600.0,8473.60,3442.400,0.00,1,0,1,1,3,3.481240,4.662684
3,60182,49.0,0,0,1,171.23,34.4,1,0,1,1,1,0,0,0,1,0,0,0,0,0,1,49.0,0.0,0.0,1685.6,8390.27,5890.312,0.00,1,1,1,0,2,3.538057,5.143008
4,1665,79.0,1,0,1,174.12,24.0,1,0,0,0,1,1,0,0,0,1,0,0,0,1,0,79.0,79.0,0.0,1896.0,13755.48,4178.880,174.12,0,1,1,1,3,3.178054,5.159745


In [38]:
df.shape

(5110, 36)

## Model Building

### Splitting to training, validation, and testing sets

In [39]:
columns_name = columns_name = [col for col in df.columns if col != 'id']

In [40]:
X = df[columns_name]
X.drop('stroke', axis=1, inplace=True)
y = df['stroke']

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_644\91966498.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.drop('stroke', axis=1, inplace=True)


In [41]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.5, random_state = 42, stratify = y_test)

### Create base classifiers

In [42]:
base_classifier_rf = RandomForestClassifier(random_state=42)

In [43]:
base_classifier_xgb = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

In [44]:
base_classifier_lgbm = lgb.LGBMClassifier(random_state=42, verbose=-1)

### Fitting base classifier on unsampled data

In [45]:
retained_columns_rf = ['avg_glucose_level', 'age', 'bmi', 'age_ever_married', 'risk_count', 'age_hypertension', 'age_heart_disease', 'urban', 'male', 'work_type_private']

In [46]:
retained_columns_xgb = ['age', 'smoking_status_unknown', 'ever_married', 'smoking_status_smokes', 'smoking_status_never_smoked', 'age_heart_disease', 'bmi', 'age_hypertension', 'work_type_private', 'age_ever_married']

In [47]:
# retained_columns_xgb = ['age', 'ever_married', 'age_bmi', 'smoking_status_smokes', 'age_hypertension', 'bmi', 'work_type_govt_job', 'smoking_status_never_smoked', 'risk_count', 'avg_glucose_level']

In [48]:
retained_columns_lgbm = ['avg_glucose_level', 'age', 'bmi', 'age_ever_married', 'urban', 'hypertension', 'male', 'work_type_self_employed', 'smoke', 'smoking_status_never_smoked']

In [49]:
base_classifier_rf.fit(X_train[retained_columns_rf], y_train)

RandomForestClassifier(random_state=42)

In [50]:
importances_rf = base_classifier_rf.feature_importances_

In [51]:
feature_imp_df_rf = pd.DataFrame({'Feature': retained_columns_rf, 'RandomForest': importances_rf}).sort_values('RandomForest', ascending=False)

In [52]:
print(feature_imp_df_rf)

             Feature  RandomForest
0  avg_glucose_level      0.260621
2                bmi      0.215085
1                age      0.143324
3   age_ever_married      0.115840
4         risk_count      0.060717
5   age_hypertension      0.052116
6  age_heart_disease      0.043564
9  work_type_private      0.037957
7              urban      0.035783
8               male      0.034991


In [53]:
base_classifier_xgb.fit(X_train[retained_columns_xgb], y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [54]:
importances_xgb = base_classifier_xgb.feature_importances_

In [55]:
feature_imp_df_xgb = pd.DataFrame({'Feature': retained_columns_xgb, 'XGBoost': importances_xgb}).sort_values('XGBoost', ascending=False)

In [56]:
print(feature_imp_df_xgb)

                       Feature   XGBoost
0                          age  0.212784
3        smoking_status_smokes  0.129538
5            age_heart_disease  0.111614
9             age_ever_married  0.085642
6                          bmi  0.082459
2                 ever_married  0.082144
7             age_hypertension  0.079435
8            work_type_private  0.076071
4  smoking_status_never_smoked  0.070581
1       smoking_status_unknown  0.069731


In [57]:
base_classifier_lgbm.fit(X_train[retained_columns_lgbm], y_train)

LGBMClassifier(random_state=42, verbose=-1)

In [58]:
importances_lgbm = base_classifier_lgbm.feature_importances_

In [59]:
feature_imp_df_lgbm = pd.DataFrame({'Feature': retained_columns_lgbm, 'LightGBM': importances_lgbm}).sort_values('LightGBM', ascending=False)

In [60]:
print(feature_imp_df_lgbm)

                       Feature  LightGBM
0            avg_glucose_level       952
2                          bmi       880
1                          age       528
3             age_ever_married       253
5                 hypertension        73
8                        smoke        73
6                         male        71
4                        urban        67
7      work_type_self_employed        56
9  smoking_status_never_smoked        47


In [61]:
feature_imp_df = pd.merge(feature_imp_df_rf, feature_imp_df_xgb, on="Feature", how="outer")

In [62]:
feature_imp_df = pd.merge(feature_imp_df, feature_imp_df_lgbm, on="Feature", how="outer")

In [63]:
feature_imp_df = feature_imp_df.fillna(0)

In [64]:
print(feature_imp_df)

                        Feature  RandomForest   XGBoost  LightGBM
0                           age      0.143324  0.212784     528.0
1              age_ever_married      0.115840  0.085642     253.0
2             age_heart_disease      0.043564  0.111614       0.0
3              age_hypertension      0.052116  0.079435       0.0
4             avg_glucose_level      0.260621  0.000000     952.0
5                           bmi      0.215085  0.082459     880.0
6                  ever_married      0.000000  0.082144       0.0
7                  hypertension      0.000000  0.000000      73.0
8                          male      0.034991  0.000000      71.0
9                    risk_count      0.060717  0.000000       0.0
10                        smoke      0.000000  0.000000      73.0
11  smoking_status_never_smoked      0.000000  0.070581      47.0
12        smoking_status_smokes      0.000000  0.129538       0.0
13       smoking_status_unknown      0.000000  0.069731       0.0
14        

In [65]:
cols_to_scale = ["RandomForest", "XGBoost", "LightGBM"]

In [66]:
scaler = MinMaxScaler()

In [67]:
feature_imp_df[cols_to_scale] = scaler.fit_transform(feature_imp_df[cols_to_scale])

In [68]:
print(feature_imp_df)

                        Feature  RandomForest   XGBoost  LightGBM
0                           age      0.549933  1.000000  0.554622
1              age_ever_married      0.444478  0.402484  0.265756
2             age_heart_disease      0.167156  0.524539  0.000000
3              age_hypertension      0.199970  0.373311  0.000000
4             avg_glucose_level      1.000000  0.000000  1.000000
5                           bmi      0.825278  0.387524  0.924370
6                  ever_married      0.000000  0.386043  0.000000
7                  hypertension      0.000000  0.000000  0.076681
8                          male      0.134261  0.000000  0.074580
9                    risk_count      0.232970  0.000000  0.000000
10                        smoke      0.000000  0.000000  0.076681
11  smoking_status_never_smoked      0.000000  0.331704  0.049370
12        smoking_status_smokes      0.000000  0.608776  0.000000
13       smoking_status_unknown      0.000000  0.327708  0.000000
14        

In [69]:
feature_imp_df["MeanImportance"] = feature_imp_df[cols_to_scale].mean(axis=1)

In [70]:
feature_imp_df = feature_imp_df.sort_values("MeanImportance", ascending=False)

In [71]:
print(feature_imp_df)

                        Feature  RandomForest   XGBoost  LightGBM  \
5                           bmi      0.825278  0.387524  0.924370   
0                           age      0.549933  1.000000  0.554622   
4             avg_glucose_level      1.000000  0.000000  1.000000   
1              age_ever_married      0.444478  0.402484  0.265756   
2             age_heart_disease      0.167156  0.524539  0.000000   
12        smoking_status_smokes      0.000000  0.608776  0.000000   
3              age_hypertension      0.199970  0.373311  0.000000   
15            work_type_private      0.145642  0.357505  0.000000   
6                  ever_married      0.000000  0.386043  0.000000   
11  smoking_status_never_smoked      0.000000  0.331704  0.049370   
13       smoking_status_unknown      0.000000  0.327708  0.000000   
9                    risk_count      0.232970  0.000000  0.000000   
8                          male      0.134261  0.000000  0.074580   
14                        urban   

In [72]:
# retained_columns = ['bmi', 'age', 'avg_glucose_level', 'age_ever_married', 'age_heart_disease', 'smoking_status_smokes', 'age_hypertension', 'work_type_private']

In [73]:
# retained_columns = ['bmi', 'age', 'avg_glucose_level', 'age_ever_married', 'age_heart_disease', 'smoking_status_smokes', 'age_hypertension', 'work_type_private', 'ever_married']

In [74]:
# retained_columns = ['bmi', 'age', 'avg_glucose_level', 'age_ever_married', 'age_heart_disease', 'smoking_status_smokes', 'age_hypertension', 'work_type_private', 'ever_married', 'smoking_status_never_smoked']

In [75]:
# retained_columns = ['bmi', 'age', 'avg_glucose_level', 'age_ever_married', 'age_heart_disease', 'smoking_status_smokes', 'age_hypertension', 'work_type_private', 'ever_married', 'smoking_status_never_smoked', 'smoking_status_unknown']

In [76]:
retained_columns = ['bmi', 'age', 'avg_glucose_level', 'age_ever_married', 'age_heart_disease', 'smoking_status_smokes', 'age_hypertension', 'work_type_private', 'ever_married', 'smoking_status_never_smoked', 'smoking_status_unknown', 'risk_count']

In [77]:
# retained_columns = ['bmi', 'age', 'avg_glucose_level', 'age_ever_married', 'age_heart_disease', 'smoking_status_smokes', 'age_hypertension', 'work_type_private', 'ever_married', 'smoking_status_never_smoked', 'smoking_status_unknown', 'risk_count', 'male']

In [78]:
# retained_columns = ['avg_glucose_level', 'bmi', 'age', 'smoking_status_smokes', 'risk_count', 'age_bmi', 'age_ever_married', 'age_hypertension']

In [79]:
# retained_columns = ['avg_glucose_level', 'bmi', 'age', 'smoking_status_smokes', 'risk_count', 'age_bmi', 'age_ever_married', 'age_hypertension', 'work_type_govt_job']

In [80]:
# retained_columns = ['avg_glucose_level', 'bmi', 'age', 'smoking_status_smokes', 'risk_count', 'age_bmi', 'age_ever_married', 'age_hypertension', 'work_type_govt_job', 'smoking_status_never_smoked']

In [81]:
X_train_retained = X_train[retained_columns]

In [82]:
X_val_retained = X_val[retained_columns]

In [83]:
best_params_rf = {'n_estimators': 221, 'max_depth': 24, 'max_leaf_nodes': 117, 'min_samples_split': 24, 'min_samples_leaf': 2, 'min_weight_fraction_leaf': 0.014351282275609447, 'max_features': 0.7, 'max_samples': 0.7350710798701602, 'criterion': 'entropy', 'min_impurity_decrease': 0.0028242087483375524, 'ccp_alpha': 0.002044925561392647, 'class_weight': 'balanced'}

In [84]:
# best_params_xgb = {'max_depth': 8, 'learning_rate': 0.05455471168883755, 'n_estimators': 186, 'min_child_weight': 7, 'max_delta_step': 9, 'subsample': 0.9150538744186808, 'colsample_bytree': 0.4776895536294967, 'colsample_bylevel': 0.8074122734272388, 'colsample_bynode': 0.5731967866429041, 'gamma': 0.17409349437318175, 'reg_alpha': 0.017374679636583235, 'reg_lambda': 7.834188155939701, 'scale_pos_weight': 10.611449136206812, 'tree_method': 'hist', 'grow_policy': 'depthwise', 'max_leaves': 105, 'min_split_loss': 1.4238466980269138, 'max_bin': 170}

In [85]:
# best_params_xgb = {'max_depth': 5, 'learning_rate': 0.03520480857681117, 'n_estimators': 435, 'min_child_weight': 8, 'max_delta_step': 5, 'subsample': 0.9791048097162386, 'colsample_bytree': 0.43131910940632695, 'colsample_bylevel': 0.6218391320943735, 'colsample_bynode': 0.7264444169815444, 'gamma': 3.821183985272892, 'reg_alpha': 4.90377432233924, 'reg_lambda': 7.138285171165027, 'scale_pos_weight': 10.925279115580084, 'tree_method': 'approx', 'grow_policy': 'depthwise', 'max_leaves': 252, 'min_split_loss': 2.9688592540894785, 'max_bin': 467}

In [86]:
best_params_xgb = {'max_depth': 3, 'learning_rate': 0.061243218939747816, 'n_estimators': 202, 'min_child_weight': 10, 'max_delta_step': 2, 'subsample': 0.8903630930892706, 'colsample_bytree': 0.4292008187481639, 'colsample_bylevel': 0.5658944131108594, 'colsample_bynode': 0.7029454459663873, 'gamma': 4.7185045808007775, 'reg_alpha': 6.5586250064674925, 'reg_lambda': 7.597183663450712, 'scale_pos_weight': 12.899902469142145, 'tree_method': 'approx', 'grow_policy': 'lossguide', 'max_leaves': 30, 'min_split_loss': 2.7703121132333073, 'max_bin': 405}

In [87]:
best_params_lgbm = {'num_leaves': 118, 'max_depth': 8, 'min_child_samples': 34, 'min_child_weight': 0.0019900294648756926, 'min_split_gain': 2.0411153790841694, 'learning_rate': 0.006957001076177289, 'n_estimators': 495, 'subsample': 0.7708065302692244, 'subsample_freq': 5, 'colsample_bytree': 0.7646621089769187, 'reg_alpha': 0.11567144028375737, 'reg_lambda': 2.91568675362154, 'max_delta_step': 1.7246538927085886, 'scale_pos_weight': 13.54579054162353, 'max_bin': 373}

In [88]:
base_estimators = [
    ('rf', RandomForestClassifier(**best_params_rf, random_state=42)),
    ('xgb', xgb.XGBClassifier(**best_params_xgb, random_state=42)),
    ('lgbm', lgb.LGBMClassifier(**best_params_lgbm, random_state=42, verbose=-1))
]

In [89]:
final_classifier = VotingClassifier(
    estimators=base_estimators,
    voting='soft'
)

In [90]:
final_classifier.fit(X_train_retained, y_train)

VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(ccp_alpha=0.002044925561392647,
                                                     class_weight='balanced',
                                                     criterion='entropy',
                                                     max_depth=24,
                                                     max_features=0.7,
                                                     max_leaf_nodes=117,
                                                     max_samples=0.7350710798701602,
                                                     min_impurity_decrease=0.0028242087483375524,
                                                     min_samples_leaf=2,
                                                     min_samples_split=24,
                                                     min_weight_fraction_leaf=0.014351282275609447,
                                                     n_estim...
                                             max_delta_step=1.7246538927085886,
                                             max_depth=8, min_child_samples=34,
                                             min_child_weight=0.0019900294648756926,
                                             min_split_gain=2.0411153790841694,
                                             n_estimators=495, num_leaves=118,
                                             random_state=42,
                                             reg_alpha=0.11567144028375737,
                                             reg_lambda=2.91568675362154,
                                             scale_pos_weight=13.54579054162353,
                                             subsample=0.7708065302692244,
                                             subsample_freq=5, verbose=-1))],
                 voting='soft')

In [91]:
y_pred_val = final_classifier.predict(X_val_retained)

In [92]:
y_proba_val = final_classifier.predict_proba(X_val_retained)[:, 1]

In [93]:
print(confusion_matrix(y_val, y_pred_val))

[[615 114]
 [ 13  24]]


In [94]:
print(accuracy_score(y_val, y_pred_val))

0.8342036553524804


In [95]:
print(precision_score(y_val, y_pred_val))

0.17391304347826086


In [96]:
print(recall_score(y_val, y_pred_val))

0.6486486486486487


In [97]:
print(fbeta_score(y_val, y_pred_val, beta=2))

0.4195804195804196


### Threshold tuning

In [98]:
best_threshold = 0.5

In [99]:
best_precision_score = 0

In [100]:
best_recall_score = 0

In [101]:
best_f2_score = 0

In [102]:
for threshold in np.arange(0.1, 0.95, 0.05):
    y_pred_val_threshold = (y_proba_val >= threshold).astype(int)
    recall = recall_score(y_val, y_pred_val_threshold)
    precision = precision_score(y_val, y_pred_val_threshold)
    f2_score = fbeta_score(y_val, y_pred_val_threshold, beta=2)
    
    print(f'Threshold {threshold} has F2 score {f2_score}')
    
    if precision >= 0.15:
        # current_recall_score = recall
        # if current_recall_score > best_recall_score:
        #     best_recall_score = current_recall_score
        #     best_precision_score = precision
        #     best_threshold = threshold
        # elif current_recall_score == best_recall_score and precision > best_precision_score:
        #     best_precision_score = precision
        #     best_threshold = threshold
        
        current_f2_score = f2_score
        if current_f2_score > best_f2_score:
            best_f2_score = current_f2_score
            best_threshold = threshold 

Threshold 0.1 has F2 score 0.32629558541266795
Threshold 0.15000000000000002 has F2 score 0.3402061855670103
Threshold 0.20000000000000004 has F2 score 0.36026200873362446
Threshold 0.25000000000000006 has F2 score 0.38018433179723504
Threshold 0.30000000000000004 has F2 score 0.37897310513447435
Threshold 0.3500000000000001 has F2 score 0.41333333333333333
Threshold 0.40000000000000013 has F2 score 0.41055718475073316
Threshold 0.45000000000000007 has F2 score 0.4272151898734177
Threshold 0.5000000000000001 has F2 score 0.4195804195804196
Threshold 0.5500000000000002 has F2 score 0.4247104247104247
Threshold 0.6000000000000002 has F2 score 0.4219409282700422
Threshold 0.6500000000000001 has F2 score 0.4186046511627907
Threshold 0.7000000000000002 has F2 score 0.2879581151832461
Threshold 0.7500000000000002 has F2 score 0.17543859649122806
Threshold 0.8000000000000002 has F2 score 0.06622516556291391
Threshold 0.8500000000000002 has F2 score 0.0
Threshold 0.9000000000000002 has F2 scor

c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [103]:
print(best_threshold)

0.45000000000000007


In [104]:
print(best_f2_score)

0.4272151898734177


### Retrain on training + validation set

In [105]:
X_train_full = pd.concat([X_train_retained, X_val_retained], axis=0, ignore_index=True)
y_train_full = pd.concat([y_train, y_val], axis=0, ignore_index=True)

In [106]:
final_classifier.fit(X_train_full, y_train_full)

VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(ccp_alpha=0.002044925561392647,
                                                     class_weight='balanced',
                                                     criterion='entropy',
                                                     max_depth=24,
                                                     max_features=0.7,
                                                     max_leaf_nodes=117,
                                                     max_samples=0.7350710798701602,
                                                     min_impurity_decrease=0.0028242087483375524,
                                                     min_samples_leaf=2,
                                                     min_samples_split=24,
                                                     min_weight_fraction_leaf=0.014351282275609447,
                                                     n_estim...
                                             max_delta_step=1.7246538927085886,
                                             max_depth=8, min_child_samples=34,
                                             min_child_weight=0.0019900294648756926,
                                             min_split_gain=2.0411153790841694,
                                             n_estimators=495, num_leaves=118,
                                             random_state=42,
                                             reg_alpha=0.11567144028375737,
                                             reg_lambda=2.91568675362154,
                                             scale_pos_weight=13.54579054162353,
                                             subsample=0.7708065302692244,
                                             subsample_freq=5, verbose=-1))],
                 voting='soft')

### Evaluate on test set

In [107]:
X_test_retained = X_test[retained_columns]

In [108]:
y_pred_test = final_classifier.predict(X_test_retained)

In [109]:
y_proba_test = final_classifier.predict_proba(X_test_retained)[:, 1]

In [110]:
print(confusion_matrix(y_test, y_pred_test))

[[616 113]
 [  8  30]]


In [111]:
print(accuracy_score(y_test, y_pred_test))

0.8422425032594524


In [112]:
print(precision_score(y_test, y_pred_test))

0.2097902097902098


In [113]:
print(recall_score(y_test, y_pred_test))

0.7894736842105263


In [114]:
print(fbeta_score(y_test, y_pred_test, beta=2))

0.5084745762711864


### Test with best threshold 0.55

In [115]:
y_pred_test_threshold = (y_proba_test >= best_threshold).astype(int)

In [116]:
print(confusion_matrix(y_test, y_pred_test_threshold))

[[585 144]
 [  7  31]]


In [117]:
print(accuracy_score(y_test, y_pred_test_threshold))

0.803129074315515


In [118]:
print(precision_score(y_test, y_pred_test_threshold))

0.17714285714285713


In [119]:
print(recall_score(y_test, y_pred_test_threshold))

0.8157894736842105


In [120]:
print(fbeta_score(y_test, y_pred_test_threshold, beta=2))

0.4740061162079511
